# Setup

In [ ]:
!sudo apt update -qq 
!sudo apt upgrade -y -qq
!sudo add-apt-repository -y ppa:graphics-drivers/ppa
!sudo apt update -qq
!ubuntu-drivers devices

In [ ]:
!pip install torch --quiet

In [ ]:
import torch

torch.cuda.is_available()

In [ ]:
file_content="""
keyboard-configuration  keyboard-configuration/model  select  Generic 105-key (Intl) PC
keyboard-configuration  keyboard-configuration/layout select  English (US)
keyboard-configuration  keyboard-configuration/variant select  English (US)
keyboard-configuration  keyboard-configuration/optionscode string
"""

conffile = open("/etc/default/keyboard-configuration", "w")
conffile.write(file_content)
conffile.close()

!sudo debconf-set-selections < /etc/default/keyboard-configuration

!sudo DEBIAN_FRONTEND=noninteractive dpkg --configure -a


In [ ]:
!sudo apt install cuda -qq


In [ ]:
!sudo apt install -y -qq  nvidia-driver-470
#restart runtime now

In [ ]:
!sudo modprobe nvidia


In [ ]:
!nvcc --version

In [ ]:
!nvcc  --list-gpu-arch

In [ ]:
!wget https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2004/x86_64/cuda-ubuntu2004.pin
!sudo mv cuda-ubuntu2004.pin /etc/apt/preferences.d/cuda-repository-pin-600
!sudo apt-key adv --fetch-keys https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2004/x86_64/7fa2af80.pub
!sudo add-apt-repository -y "deb http://developer.download.nvidia.com/compute/cuda/repos/ubuntu2004/x86_64/ /"
!sudo apt-get update -qq
!sudo apt-get -y install cuda -qq


In [ ]:
!sudo apt-get install pciutils -qq

In [ ]:
!lspci | grep -i vga

In [ ]:
import torch
torch.cuda.is_available()
torch.cuda.get_device_name(0)


In [ ]:
!nvidia-smi

# Section 3.1

In [ ]:
!pip install transformers  google --quiet

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification

model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)



In [ ]:
!pip install openai google SentencePiece  --q

In [ ]:
from openai import OpenAI
#from google.colab import userdata

#client = OpenAI(api_key=userdata.get('open_ai_key'))
client = OpenAI(api_key="<YOUR KEY HERE>")

prompt="Translate the following English text to French: 'Hello, how are you?'",

response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that translates text."},
            {"role": "user", "content": str(prompt) }
        ],
        max_tokens=150,
        temperature=0.7,
    )

print(response.choices[0].message.content.strip())


In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = 't5-base'
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)


In [ ]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification

model_name = 'roberta-base'
tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = RobertaForSequenceClassification.from_pretrained(model_name)


# Section 3.2

In [ ]:
pip install torch torchvision datasets -q

In [ ]:
import torch.nn as nn

# Define a simple neural network model
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.fc1 = nn.Linear(64*7*7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.max_pool2d(x, 2)
        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2)
        x = x.view(-1, 64*7*7)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

def create_model(learning_rate, batch_size=5):
    model = SimpleCNN().cuda()
    return model


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.amp import GradScaler, autocast

# Load MNIST dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Initialize model, loss function, and optimizer
model = create_model(learning_rate=2e-5)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Initialize GradScaler for mixed-precision training
scaler = GradScaler('cuda')

# Training loop
for epoch in range(1):  # Training for 1 epoch for demonstration
    correct = 0
    total = 0
    running_loss = 0.0

    for data, target in train_loader:
        data, target = data.cuda(), target.cuda()  # Move data to GPU

        optimizer.zero_grad()

        # Forward pass with autocast for mixed-precision
        with autocast('cuda'):
            output = model(data)
            loss = criterion(output, target)

        # Backward pass and optimization
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Calculate running loss
        running_loss += loss.item()

        # Calculate accuracy
        _, predicted = torch.max(output.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = 100 * correct / total

    print(f'Epoch [{epoch+1}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.2f}%')



In [ ]:
!pip install optuna numpy==1.22.4 -q

In [ ]:
import optuna
import torch
from datasets import load_dataset
from transformers import Trainer, TrainingArguments
from transformers import BertTokenizer, BertForSequenceClassification
import numpy as np

model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": (predictions == labels).mean()} # Define accuracy computation

def preprocess(data):
   return tokenizer(data['sentence1'], data['sentence2'], truncation=True, padding='max_length', max_length=100)

def objective(trial):
    lr = trial.suggest_float('lr', 1e-5, 1e-1, log=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Load datasets
    dataset_train = load_dataset('glue', 'mrpc', split='train')
    dataset_validation = load_dataset('glue', 'mrpc', split='validation') # Load validation dataset

    # Apply the preprocessing to the datasets
    dataset_train = dataset_train.map(preprocess, batched=True)
    dataset_validation = dataset_validation.map(preprocess, batched=True) # Preprocess validation dataset

    # Remove the unnecessary columns added by the tokenizer
    dataset_train = dataset_train.remove_columns(['sentence1', 'sentence2'])
    dataset_validation = dataset_validation.remove_columns(['sentence1', 'sentence2']) # Remove columns from validation dataset

    # Ensure the format is correct for PyTorch
    dataset_train.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
    dataset_validation.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label']) # Set format for validation dataset

    # Suggest hyperparameters
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)

    # Define training arguments
    training_args = TrainingArguments(
        output_dir='./results',
        eval_strategy='epoch',
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=3,
        weight_decay=weight_decay,
    )

    # Initialize Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset_train,
        eval_dataset=dataset_validation,
        compute_metrics=compute_metrics, # Pass the compute_metrics function
    )

    # Train and evaluate
    trainer.train()
    eval_result = trainer.evaluate()
    return eval_result['eval_accuracy'] # Now eval_result should contain 'eval_accuracy'


study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=5) #recommend 100 or so
print(study.best_params)


In [ ]:
!yes|pip uninstall numpy -q
!pip install numpy==1.26.0 -q
!pip install --upgrade datasets==2.19.2 transformers==4.41.2 -q
!pip install pandas==2.0.3 -q
# you'll need to restart the runtime after this

In [ ]:
pip freeze

In [ ]:
!pip install torch torchvision torchaudio -q
!pip install transformers[torch] -q
!pip install datasets -q


In [ ]:
!yes|pip uninstall pandas --quiet
!pip install pandas --quiet
!pip install --upgrade numpy pyarrow --quiet


In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
from datasets import load_dataset

# Load pre-trained model and tokenizer
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)

# Load dataset
dataset = load_dataset('glue', 'mrpc', split='train')


In [ ]:
!pip install accelerate>=0.21.0
# you'll need to restart the runtime after this

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
from datasets import load_dataset

# Load pre-trained model and tokenizer
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)

# Load dataset
dataset = load_dataset('glue', 'mrpc', split='train')
dataset_validation = load_dataset('glue', 'mrpc', split='validation') # Load validation dataset


In [ ]:
def preprocess(data):
   return tokenizer(data['sentence1'], data['sentence2'], truncation=True, padding='max_length', max_length=100)

dataset = dataset.map(preprocess, batched=True)
dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
dataset_validation = dataset_validation.map(preprocess, batched=True)
dataset_validation.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label']) # Set format for validation dataset#

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
   output_dir='./results',
   eval_strategy='epoch',
   learning_rate=2e-5,
   per_device_train_batch_size=8,
   per_device_eval_batch_size=8,
   num_train_epochs=3,
   weight_decay=0.01,
)

trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=dataset,
   eval_dataset=dataset_validation,
)

trainer.train()
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
from datasets import load_dataset

# Load pre-trained model and tokenizer
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)

# Load datasets
dataset_train = load_dataset('glue', 'mrpc', split='train')
dataset_validation = load_dataset('glue', 'mrpc', split='validation') # Load validation dataset

def preprocess(data):
    # Correctly apply padding and return as PyTorch tensors
    return tokenizer(data['sentence1'], data['sentence2'], truncation=True, padding='max_length', max_length=100, return_tensors='pt')

# Apply the preprocessing to the datasets
dataset_train = dataset_train.map(preprocess, batched=True)
dataset_validation = dataset_validation.map(preprocess, batched=True) # Preprocess validation dataset

# Remove the unnecessary columns added by the tokenizer
dataset_train = dataset_train.remove_columns(['sentence1', 'sentence2'])
dataset_validation = dataset_validation.remove_columns(['sentence1', 'sentence2']) # Remove columns from validation dataset

# Ensure the format is correct for PyTorch
dataset_train.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
dataset_validation.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label']) # Set format for validation dataset

from transformers import Trainer, TrainingArguments

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train, # Pass training dataset
    eval_dataset=dataset_validation # Pass validation dataset
)

# Train the model
trainer.train()
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

In [ ]:
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")


In [ ]:
!pip install optuna --quiet

In [ ]:
import optuna
from transformers import Trainer, TrainingArguments
import numpy as np
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification

# Load pre-trained model and tokenizer
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": (predictions == labels).mean()} # Define accuracy computation

def preprocess(data):
    # Correctly apply padding and return as PyTorch tensors
    return tokenizer(data['sentence1'], data['sentence2'], truncation=True, padding='max_length', max_length=100, return_tensors='pt')

def objective(trial):
    # Load datasets
    dataset_train = load_dataset('glue', 'mrpc', split='train')
    dataset_validation = load_dataset('glue', 'mrpc', split='validation') # Load validation dataset

    # Apply the preprocessing to the datasets
    dataset_train = dataset_train.map(preprocess, batched=True)
    dataset_validation = dataset_validation.map(preprocess, batched=True) # Preprocess validation dataset

    # Remove the unnecessary columns added by the tokenizer
    dataset_train = dataset_train.remove_columns(['sentence1', 'sentence2'])
    dataset_validation = dataset_validation.remove_columns(['sentence1', 'sentence2']) # Remove columns from validation dataset

    # Ensure the format is correct for PyTorch
    dataset_train.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
    dataset_validation.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label']) # Set format for validation dataset

    # Suggest hyperparameters
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)

    # Define training arguments
    training_args = TrainingArguments(
        output_dir='./results',
        eval_strategy='epoch',
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=3,
        weight_decay=weight_decay,
    )

    # Initialize Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset_train,
        eval_dataset=dataset_validation,
        compute_metrics=compute_metrics, # Pass the compute_metrics function
    )

    # Train and evaluate
    trainer.train()
    eval_result = trainer.evaluate()
    return eval_result['eval_accuracy'] # Now eval_result should contain 'eval_accuracy'

# Create study and optimize
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=5) #50 trials would be more useful

print(f"Best hyperparameters: {study.best_params}")

# Section 3.3

Working Grid Search Below


In [ ]:
!pip install scikit-learn --quiet

In [ ]:
from sklearn.base import BaseEstimator
from sklearn.model_selection import GridSearchCV, KFold, train_test_split  # Import KFold here
from transformers import Trainer, TrainingArguments
from sklearn.tree import DecisionTreeRegressor  # Import DecisionTreeRegressor for continuous targets
from sklearn.ensemble import BaggingRegressor # Import BaggingRegressor for continuous targets
from sklearn.datasets import make_regression

# Generate a regression dataset with 1000 samples and 10 features
X, y = make_regression(n_samples=1000, n_features=10, noise=0.1, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

param_grid = {
    'n_estimators': [100, 500, 1000],
    'max_samples': [0.5, 0.75, 1.0],
    'max_features': [0.5, 0.75, 1.0],
    'bootstrap': [True, False],
    'bootstrap_features': [True, False]
}
# Use DecisionTreeRegressor and BaggingRegressor for continuous target
clf = DecisionTreeRegressor(criterion='squared_error', max_features='sqrt')
classifier = BaggingRegressor(estimator=clf, n_estimators=2000, max_features=1.0,
                         max_samples=.5, oob_score=True, n_jobs=1)

# Define a 2-fold cross-validator
cv = KFold(n_splits=2, shuffle=True, random_state=42)

# Use GridSearchCV with the custom estimator
grid_search = GridSearchCV(
    estimator=classifier,  # Pass your custom estimator
    param_grid=param_grid,
    # Removed scoring parameter since the estimator now has a score method
    refit=True,
    cv=cv
)

# Pass the param_grid to fit, as it contains the hyperparameters to search
grid_search.fit(X_train, y_train)

# Get the best hyperparameters
best_params = grid_search.best_params_
print(f"Best hyperparameters: {best_params}")

In [ ]:
# Get the best hyperparameters
best_params = grid_search.best_params_
print(f"Best hyperparameters: {best_params}")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.base import BaseEstimator
from sklearn.model_selection import KFold, train_test_split  # Import KFold here
from transformers import Trainer, TrainingArguments
from sklearn.tree import DecisionTreeRegressor  # Import DecisionTreeRegressor for continuous targets
from sklearn.ensemble import BaggingRegressor # Import BaggingRegressor for continuous targets
from scipy.stats import randint
# ... (your existing code for param_grid, train_and_evaluate)
from sklearn.datasets import make_regression

# Generate a regression dataset with 1000 samples and 10 features
X, y = make_regression(n_samples=1000, n_features=10, noise=0.1, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

param_distributions = {
    'n_estimators': [100, 500, 1000],
    'max_samples': [0.5, 0.75, 1.0],
    'max_features': [0.5, 0.75, 1.0],
    'bootstrap': [True, False],
    'bootstrap_features': [True, False]
}

# Use DecisionTreeRegressor and BaggingRegressor for continuous target
clf = DecisionTreeRegressor(criterion='squared_error', max_features='sqrt')
classifier = BaggingRegressor(estimator=clf, n_estimators=1000, max_features=1.,
                         max_samples=.5, oob_score=True, n_jobs=1)

random_search = RandomizedSearchCV(estimator=classifier, param_distributions=param_distributions, n_iter=10, scoring='neg_mean_squared_error',cv=5, random_state=42)
random_search.fit(X_train, y_train)
best_params = random_search.best_params_

# Get the best hyperparameters
best_params = grid_search.best_params_
print(f"Best hyperparameters: {best_params}")

In [ ]:
!pip install optuna --quiet

In [ ]:
# Example usage with RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.datasets import make_regression
from scipy.stats import randint

def train_and_evaluate_model(model, X_train, y_train, X_val, y_val):
    # Train the model
    model.fit(X_train, y_train)

    # Predict on validation set
    y_pred = model.predict(X_val)

    # Calculate evaluation metrics
    mse = mean_squared_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)

    print(f"Mean Squared Error: {mse}")
    print(f"R^2 Score: {r2}")

    return model, mse, r2

# Generate a regression dataset with 1000 samples and 10 features
X, y = make_regression(n_samples=1000, n_features=10, noise=0.1, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the parameter distributions
param_distributions = {
    'estimator__criterion': ['squared_error', 'friedman_mse', 'absolute_error'],
    'estimator__max_depth': [None, 5, 10],
    'n_estimators': randint(100, 2000)  # Using a range for n_estimators
}

# Initialize the base estimator and the BaggingRegressor
clf = DecisionTreeRegressor()
classifier = BaggingRegressor(estimator=clf, n_estimators=1000, max_features=1.0,
                              max_samples=5, oob_score=True, n_jobs=1)

# Create the RandomizedSearchCV object
random_search = RandomizedSearchCV(estimator=classifier, param_distributions=param_distributions, n_iter=10, scoring='neg_mean_squared_error', cv=5, random_state=42)

# Fit the model using RandomizedSearchCV
random_search.fit(X_train, y_train)

# Get the best model from random search
best_model = random_search.best_estimator_

# Train and evaluate the best model
best_model, mse, r2 = train_and_evaluate_model(best_model, X_train, y_train, X_val, y_val)

print(best_model, mse, r2)

In [ ]:
!pip install keras keras_tuner -q

In [ ]:
import torch.nn as nn

# Define a simple neural network model
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.fc1 = nn.Linear(64*7*7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.max_pool2d(x, 2)
        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2)
        x = x.view(-1, 64*7*7)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x



In [ ]:
!pip install keras keras-tuner tensorflow --quiet


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import keras_tuner as kt

def SimpleCNN():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    return model


def create_model(hp):
    model = SimpleCNN()
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=hp.Float('learning_rate', 1e-4, 1e-2, sampling='log')),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

tuner = kt.Hyperband(
    create_model,
    objective='val_accuracy',
    max_epochs=5, #often more like 10
    hyperband_iterations=1, #often more like 3
    factor=3,
    directory='my_dir',
    project_name='helloworld'
)

# Example data (replace with your actual data)
(X_train, y_train), (X_val, y_val) = tf.keras.datasets.mnist.load_data()
X_train = X_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_val = X_val.reshape(-1, 28, 28, 1).astype('float32') / 255.0

# Run the hyperparameter search
tuner.search(X_train, y_train, epochs=10, validation_data=(X_val, y_val))

# Get the best hyperparameters
best_params = tuner.get_best_hyperparameters(num_trials=1)[0]
print(best_params)



In [ ]:
import torch
import torch.nn as nn
from transformers import BertModel, BertTokenizer, Trainer, TrainingArguments
from datasets import load_dataset
from transformers import EarlyStoppingCallback, default_data_collator

# Custom BERT-based model with additional dropout layers
class CustomBertModel(nn.Module):
    def __init__(self, model_name='bert-base-uncased', dropout_prob=0.3):
        super(CustomBertModel, self).__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout_prob)  # Dropout rate is set here
        self.classifier = nn.Linear(self.bert.config.hidden_size, 2)  # Assuming binary classification

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs[1]  # The pooled output from the BERT model
        pooled_output = self.dropout(pooled_output)  # Apply dropout
        logits = self.classifier(pooled_output)
        return logits

# Load tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Load datasets
dataset_train = load_dataset('glue', 'mrpc', split='train')
dataset_val = load_dataset('glue', 'mrpc', split='validation')

# Tokenize the datasets
def preprocess(data):
    return tokenizer(data['sentence1'], data['sentence2'], truncation=True, padding='max_length', max_length=100)

dataset_train = dataset_train.map(preprocess, batched=True)
dataset_val = dataset_val.map(preprocess, batched=True)

dataset_train.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
dataset_val.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# Initialize custom model with specified dropout rate
model = CustomBertModel(dropout_prob=0.3).to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    evaluation_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    save_steps=10,
    load_best_model_at_end=True,
    fp16=True  # Enable mixed-precision training
)

# Define the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_val,
    data_collator=default_data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# Train the model
trainer.train()

# Evaluate the model
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")


# Section 3.4 Principles of PEFT (Parameter Efficient Fine-Tuning)

Adapter Modules

In [ ]:
from transformers import BertModel, BertConfig
from torch import nn

class Adapter(nn.Module):
   def __init__(self, input_dim, bottleneck_dim):
       super(Adapter, self).__init__()
       self.down_project = nn.Linear(input_dim, bottleneck_dim)
       self.up_project = nn.Linear(bottleneck_dim, input_dim)
       self.activation = nn.ReLU()

   def forward(self, x):
       return self.up_project(self.activation(self.down_project(x)))

class BertWithAdapters(nn.Module):
   def __init__(self, model_name, bottleneck_dim):
       super(BertWithAdapters, self).__init__()
       self.bert = BertModel.from_pretrained(model_name)
       self.adapters = nn.ModuleList([Adapter(self.bert.config.hidden_size, bottleneck_dim) for _ in range(self.bert.config.num_hidden_layers)])

   def forward(self, input_ids, attention_mask=None):
       outputs = self.bert(input_ids, attention_mask=attention_mask)
       for i, adapter in enumerate(self.adapters):
          outputs.last_hidden_state[:, i, :] = adapter(outputs.last_hidden_state[:, i, :])
          return outputs

model_name = 'bert-base-uncased'
bottleneck_dim = 64
model = BertWithAdapters(model_name, bottleneck_dim)


Layer Freezing

In [ ]:
import torch
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained('bert-base-uncased')

# Freeze all layers except the classifier
for param in model.bert.parameters():
   param.requires_grad = False

# Fine-tune the classifier layers
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=2e-5)


L2 Regularization

In [ ]:
from torch.optim import AdamW
# Define optimizer with L2 regularization (weight decay)
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)


Gradient Scaling

In [ ]:
optimizer = torch.optim.AdamW([
   {'params': model.bert.encoder.layer[-1].parameters(), 'lr': 5e-5},
   {'params': model.bert.encoder.layer[-2].parameters(), 'lr': 5e-5},
   {'params': model.classifier.parameters(), 'lr': 2e-5}
])


Fine-tuning with adapters

In [ ]:
!pip install accelerate>=0.21.0 -q
!pip install datasets -q
# you'll need to restart the runtime after this

In [ ]:
!pip install datasets -q

In [ ]:
import torch
from transformers import BertForSequenceClassification, BertModel, BertTokenizer
from transformers import Trainer, TrainingArguments
from datasets import load_dataset

# Define your custom model
class BertWithAdapters(torch.nn.Module):
    def __init__(self, model_name, bottleneck_dim):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.tokenizer = BertTokenizer.from_pretrained(model_name) # Initialize the tokenizer here
        self.bottleneck = torch.nn.Linear(self.bert.config.hidden_size, bottleneck_dim)
        self.classifier = torch.nn.Linear(bottleneck_dim, 2) # Assuming binary classification

    def forward(self, input_ids, attention_mask, labels=None): # Add labels as argument
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        bottleneck_output = self.bottleneck(pooled_output)
        logits = self.classifier(bottleneck_output)

        loss = None
        if labels is not None: # Calculate loss if labels are provided
            loss_fct = torch.nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, 2), labels.view(-1)) # Assuming binary classification

        return {'loss': loss, 'logits': logits} # Return loss and logits


training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
)

model_name = 'bert-base-uncased'
bottleneck_dim = 64
model = BertWithAdapters(model_name, bottleneck_dim)

# Load both training and validation datasets
dataset_train = load_dataset('glue', 'mrpc', split='train')
dataset_val = load_dataset('glue', 'mrpc', split='validation') # Load the validation set

# Tokenize the datasets
def preprocess_function(examples):
    return model.tokenizer(examples['sentence1'], examples['sentence2'], truncation=True, padding=True)

dataset_train = dataset_train.map(preprocess_function, batched=True)
dataset_val = dataset_val.map(preprocess_function, batched=True) # Tokenize the validation set

# Add a data collator to prepare the inputs for the model
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=model.tokenizer) # Use the tokenizer associated with your BERT model

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train, # Pass the training dataset
    eval_dataset=dataset_val,   # Pass the validation dataset
    data_collator=data_collator # Pass the data collator to the Trainer
)

trainer.train()
eval_results = trainer.evaluate()
print(f"Evaluation results for task {i+1}: {eval_results}")

Fine-tuning with Layer-freezing

In [ ]:
!pip install datasets transformers --quiet

In [ ]:
import torch
from transformers import BertForSequenceClassification, BertModel, BertTokenizer, Trainer, TrainingArguments
from datasets import load_dataset

# Tokenize the datasets
def preprocess_function(examples):
    return tokenizer(
        examples['sentence1'],
        examples['sentence2'],
        truncation=True,
        return_overflowing_tokens=False, 
        padding='max_length',  # Add padding to max_length
        max_length=96          # Set the maximum sequence length
    )

model = BertForSequenceClassification.from_pretrained('bert-base-uncased')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased') # Initialize the tokenizer

# Only freeze the embeddings
for param in model.bert.embeddings.parameters():
    param.requires_grad = False

training_args = TrainingArguments(
   output_dir='./results',
   eval_strategy='epoch',
   learning_rate=2e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=3,
   weight_decay=0.01,
   logging_dir='./logs',
)

# Load both training and validation datasets
dataset_train = load_dataset('glue', 'mrpc', split='train')
dataset_val = load_dataset('glue', 'mrpc', split='validation') # Load the validation set


# Apply the preprocessing function to the datasets
dataset_train = dataset_train.map(preprocess_function, batched=True)
dataset_val = dataset_val.map(preprocess_function, batched=True)

trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=dataset_train, # Pass the training dataset
   eval_dataset=dataset_val,   # Pass the validation dataset
)

trainer.train()
eval_results = trainer.evaluate()
print(f"Evaluation results for task {i+1}: {eval_results}")

In [ ]:
import torch
from transformers import BertForSequenceClassification, BertModel, BertTokenizer
from transformers import Trainer, TrainingArguments
from datasets import load_dataset

# Define your custom model
class BertWithAdapters(torch.nn.Module):
    def __init__(self, model_name, bottleneck_dim):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.tokenizer = BertTokenizer.from_pretrained(model_name) # Initialize the tokenizer here
        self.bottleneck = torch.nn.Linear(self.bert.config.hidden_size, bottleneck_dim)
        self.classifier = torch.nn.Linear(bottleneck_dim, 2) # Assuming binary classification

    def forward(self, input_ids, attention_mask, labels=None): # Add labels as argument
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        bottleneck_output = self.bottleneck(pooled_output)
        logits = self.classifier(bottleneck_output)

        loss = None
        if labels is not None: # Calculate loss if labels are provided
            loss_fct = torch.nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, 2), labels.view(-1)) # Assuming binary classification

        return {'loss': loss, 'logits': logits} # Return loss and logits


training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
)

model_name = 'bert-base-uncased'
bottleneck_dim = 64
model = BertWithAdapters(model_name, bottleneck_dim)

# Load both training and validation datasets
dataset_train = load_dataset('glue', 'mrpc', split='train')
dataset_val = load_dataset('glue', 'mrpc', split='validation') # Load the validation set

# Tokenize the datasets
def preprocess_function(examples):
    return model.tokenizer(examples['sentence1'], examples['sentence2'], truncation=True, padding=True)

dataset_train = dataset_train.map(preprocess_function, batched=True)
dataset_val = dataset_val.map(preprocess_function, batched=True) # Tokenize the validation set

# Add a data collator to prepare the inputs for the model
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=model.tokenizer) # Use the tokenizer associated with your BERT model

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train, # Pass the training dataset
    eval_dataset=dataset_val,   # Pass the validation dataset
    data_collator=data_collator # Pass the data collator to the Trainer
)

trainer.train()
eval_results = trainer.evaluate()
print(f"Evaluation results for task {i+1}: {eval_results}")

# Section 3.5 Exploring LoRA (Low-Rank Adaptation) for Task-Specific Fine-Tuning

In [ ]:
   import torch
   import torch.nn as nn

   class LoRALayer(nn.Module):
       def __init__(self, input_dim, rank):
           super(LoRALayer, self).__init__()
           self.A = nn.Parameter(torch.randn(input_dim, rank))
           self.B = nn.Parameter(torch.randn(rank, input_dim))

       def forward(self, x):
           return x + torch.matmul(torch.matmul(x, self.A), self.B)

   class BertWithLoRA(nn.Module):
       def __init__(self, model_name, rank):
           super(BertWithLoRA, self).__init__()
           self.bert = BertModel.from_pretrained(model_name)
           self.lora_layers = nn.ModuleList([LoRALayer(self.bert.config.hidden_size, rank) for _ in range(self.bert.config.num_hidden_layers)])

       def forward(self, input_ids, attention_mask=None):
           outputs = self.bert(input_ids, attention_mask=attention_mask)
           for i, lora_layer in enumerate(self.lora_layers):
               outputs.last_hidden_state[:, i, :] = lora_layer(outputs.last_hidden_state[:, i, :])
           return outputs


In [ ]:
   from transformers import BertTokenizer, BertModel
   from datasets import load_dataset

   model_name = 'bert-base-uncased'
   tokenizer = BertTokenizer.from_pretrained(model_name)
   model = BertWithLoRA(model_name, rank=8)

   dataset = load_dataset('glue', 'mrpc', split='train')

   def preprocess(data):
       return tokenizer(data['sentence1'], data['sentence2'], truncation=True, padding=True)

   dataset = dataset.map(preprocess, batched=True)
   dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])


In [ ]:
!pip install transformers datasets --quiet

In [ ]:
import torch
from transformers import BertForSequenceClassification, BertModel, BertTokenizer
from transformers import Trainer, TrainingArguments
from datasets import load_dataset

# Define your custom model
class BertWithAdapters(torch.nn.Module):
    def __init__(self, model_name, bottleneck_dim):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.tokenizer = BertTokenizer.from_pretrained(model_name) # Initialize the tokenizer here
        self.bottleneck = torch.nn.Linear(self.bert.config.hidden_size, bottleneck_dim)
        self.classifier = torch.nn.Linear(bottleneck_dim, 2) # Assuming binary classification

    def forward(self, input_ids, attention_mask, labels=None): # Add labels as argument
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        bottleneck_output = self.bottleneck(pooled_output)
        logits = self.classifier(bottleneck_output)

        loss = None
        if labels is not None: # Calculate loss if labels are provided
            loss_fct = torch.nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, 2), labels.view(-1)) # Assuming binary classification

        return {'loss': loss, 'logits': logits} # Return loss and logits


training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
)

model_name = 'bert-base-uncased'
bottleneck_dim = 64
model = BertWithAdapters(model_name, bottleneck_dim)

# Load both training and validation datasets
dataset_train = load_dataset('glue', 'mrpc', split='train')
dataset_val = load_dataset('glue', 'mrpc', split='validation') # Load the validation set

# Tokenize the datasets
def preprocess_function(examples):
    return model.tokenizer(examples['sentence1'], examples['sentence2'], truncation=True, padding='longest') # Pad to the longest sequence in the batch

dataset_train = dataset_train.map(preprocess_function, batched=True)
dataset_val = dataset_val.map(preprocess_function, batched=True) # Tokenize the validation set

# Add a data collator to prepare the inputs for the model
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=model.tokenizer) # Use the tokenizer associated with your BERT model

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train, # Pass the training dataset
    eval_dataset=dataset_val,   # Pass the validation dataset
    data_collator=data_collator # Pass the data collator to the Trainer
)

trainer.train()
eval_results = trainer.evaluate()
print(f"Evaluation results for task {i+1}: {eval_results}")

# Section 3.6 Finding the Balance Between Efficiency and Performance

In [ ]:
from transformers import BertForSequenceClassification, Trainer, TrainingArguments

model = BertForSequenceClassification.from_pretrained('bert-base-uncased')
# Example code for pruning would depend on specific pruning libraries/methods


In [ ]:
from transformers import DistilBertModel, DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertModel.from_pretrained('distilbert-base-uncased')


In [ ]:
from transformers import BertForSequenceClassification, Trainer, TrainingArguments

model = BertForSequenceClassification.from_pretrained('bert-base-uncased')
model.half()  # Convert model weights to 16-bit


In [ ]:
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, TensorDataset
import torch
import torch.nn as nn
import torch.optim as optim

# Sample training data (replace this with your actual data)
train_data = [[1, 2, 3], [4, 5, 6]]
train_labels = [0, 1]

# Assuming 'train_data' and 'train_labels' are your training data and labels
# Keep the input data as LongTensor (integers)
train_dataset = TensorDataset(torch.tensor(train_data, dtype=torch.long), torch.tensor(train_labels))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Define a simple model (replace this with your actual model)
class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.embedding = nn.Embedding(10, 8)
        self.fc = nn.Linear(8 * 3, 2)  # Assuming input length of 3 and output classes of 2

    def forward(self, x):
        x = self.embedding(x)
        x = x.view(x.size(0), -1)
        logits = self.fc(x)
        return logits

# Initialize the model
model = SimpleModel().cuda()

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-5)

scaler = GradScaler('cuda')

for data, target in train_loader:
    optimizer.zero_grad()
    data, target = data.cuda(), target.cuda()
    with autocast('cuda'):
        output = model(data)
        # Ensure logits are in float32 before passing to criterion
        loss = criterion(output.float(), target)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    print(f'Loss: {loss.item()}')


Early Stopping

In [ ]:
import os
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, TensorDataset
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import EarlyStoppingCallback, TrainingArguments, Trainer, default_data_collator

os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# Sample training data (replace this with your actual data)
train_data = [[1, 2, 3], [4, 5, 6]]
train_labels = [0, 1]

# Sample evaluation data (replace this with your actual data)
eval_data = [[7, 8, 9], [10, 11, 12]]
eval_labels = [1, 0]

# Convert the data to a list of dictionaries
# Each dictionary represents a single data point with 'input_ids' and 'labels' keys
train_dataset_for_trainer = [
    {"input_ids": torch.tensor(data, dtype=torch.long),
     "labels": torch.tensor(label, dtype=torch.long)}  # Make sure labels are also tensors of dtype long
    for data, label in zip(train_data, train_labels)
]

eval_dataset_for_trainer = [
    {"input_ids": torch.tensor(data, dtype=torch.long),
     "labels": torch.tensor(label, dtype=torch.long)}
    for data, label in zip(eval_data, eval_labels)
]

# Define a simple model (replace this with your actual model)
class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.embedding = nn.Embedding(13, 8)  # Adjust embedding dimension for the vocabulary size
        self.fc = nn.Linear(8 * 3, 2)  # Assuming input length of 3 and output classes of 2

    def forward(self, input_ids=None, labels=None):
        # Debugging: Check input_ids and print if assertion fails
        if input_ids is not None:
            if not ((input_ids >= 0).all() and (input_ids < 13).all()):
                print("Warning: input_ids out of bounds:", input_ids)

        x = self.embedding(input_ids)
        x = x.view(x.size(0), -1)
        logits = self.fc(x)
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            # No need to reshape logits, CrossEntropyLoss expects raw logits and class indices
            loss = loss_fct(logits, labels)
        return {'loss': loss, 'logits': logits}

# Initialize the model
model = SimpleModel()
model = model.cuda()  # Comment this line to run on CPU

# Training arguments
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',  # Use the updated argument name
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir='./logs',
    load_best_model_at_end=True,
    save_strategy='epoch'
)

# Create the Trainer object
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_for_trainer,  # Use the correct dataset variable
    eval_dataset=eval_dataset_for_trainer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    data_collator=default_data_collator
)

trainer.train()
eval_results = trainer.evaluate()
print(f"Evaluation results for task {i+1}: {eval_results}")

In [ ]:
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import EarlyStoppingCallback, TrainingArguments, Trainer, default_data_collator, DistilBertTokenizer # Import the tokenizer

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
dataset = load_dataset('glue', 'mrpc', split='train')

def preprocess(data):
  return tokenizer(data['sentence1'], data['sentence2'], truncation=True, padding=True)

dataset = dataset.map(preprocess, batched=True)

train_loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)


Practical Example

1. Model Pruning and Knowledge Distillation

In [ ]:
from transformers import BertForSequenceClassification, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
from transformers import EarlyStoppingCallback, default_data_collator, DistilBertTokenizer
import torch
from torch.nn import functional as F
from torch.utils.data import DataLoader

# Define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load datasets
dataset_train = load_dataset('glue', 'mrpc', split='train')
dataset_val = load_dataset('glue', 'mrpc', split='validation')

# Load tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Tokenize the datasets
def preprocess(data):
    return tokenizer(data['sentence1'], data['sentence2'], truncation=True, padding='max_length', max_length=100)

dataset_train = dataset_train.map(preprocess, batched=True)
dataset_val = dataset_val.map(preprocess, batched=True)

dataset_train.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
dataset_val.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# Load teacher model
teacher_model = BertForSequenceClassification.from_pretrained('bert-base-uncased').to(device)

# Load student model
student_model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased').to(device)


# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    save_steps=10,
)

# Distillation loss
def distillation_loss(student_outputs, teacher_outputs, temperature=2.0):
    soft_labels = nn.functional.softmax(teacher_outputs.logits / temperature, dim=-1)
    student_logits = student_outputs.logits / temperature
    return nn.functional.kl_div(nn.functional.log_softmax(student_logits, dim=-1), soft_labels, reduction='batchmean') * (temperature ** 2)

# Define custom Trainer for distillation
class DistillationTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        student_outputs = model(**inputs)
        with torch.no_grad():
            teacher_outputs = teacher_model(**inputs)
        loss = distillation_loss(student_outputs, teacher_outputs) + nn.functional.cross_entropy(student_outputs.logits, labels)
        return (loss, student_outputs) if return_outputs else loss


# Define the Trainer
trainer = DistillationTrainer(
    model=student_model,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_val,
    data_collator=default_data_collator,
)

# Train the student model
trainer.train()
eval_results = trainer.evaluate()
print(f"Evaluation results for task {i+1}: {eval_results}")

Prune the student model

In [ ]:
import torch.nn.utils.prune as prune
from transformers import DistilBertForSequenceClassification

# Example of pruning a linear layer in a neural network
# DistilBertForSequenceClassification typically has a single classifier layer named 'classifier'
module = student_model.classifier  # Access the classifier layer directly

# Prune 40% of the weights in the classifier layer
prune.l1_unstructured(module, name='weight', amount=0.4)

# Optional: Remove the pruning reparameterization to make pruning permanent
prune.remove(module, 'weight')

# Verify the pruning
print(module.weight)


2. Mixed Precision Training and Early Stopping

In [ ]:
from transformers import BertForSequenceClassification, Trainer, TrainingArguments, default_data_collator
from torch.cuda.amp import GradScaler, autocast

model = BertForSequenceClassification.from_pretrained('bert-base-uncased')

def preprocess(data):
  return tokenizer(data['sentence1'], data['sentence2'], truncation=True, padding='max_length', max_length=100)

dataset_train = dataset_train.map(preprocess, batched=True)
dataset_val = dataset_val.map(preprocess, batched=True)

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='steps',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir='./logs',
    load_best_model_at_end=True,
    fp16=True  # Enable mixed-precision training
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_val,
    # Use the default data collator to handle variable sequence lengths
    data_collator=default_data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer.train()
eval_results = trainer.evaluate()
print(f"Evaluation results for task {i+1}: {eval_results}")


# Section 3.7 Moving beyond PEFT and LoRA

1. Few-shot Learning

In [ ]:
from openai import OpenAI
#from google.colab import userdata

#client = OpenAI(api_key=userdata.get('open_ai_key'))
client = OpenAI(api_key="<YOUR KEY HERE>")

# Define a few-shot learning prompt
prompt = """
The following is a conversation with an AI assistant. The assistant is helpful, creative, clever, and very friendly.

Human: Hello, who are you?
AI: I am an AI created by OpenAI. How can I help you today?
Human: Can you tell me a joke?
AI: Sure! Why don't scientists trust atoms? Because they make up everything!

Human: What is the capital of France?
AI: The capital of France is Paris.

Human: How does photosynthesis work?
AI: Photosynthesis is the process by which green plants and some other organisms use sunlight to synthesize foods with the help of chlorophyll from carbon dioxide and water.
"""

# Add your query to the prompt
prompt += "\nHuman: What is the largest planet in our solar system?\nAI:"


response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that answers questions."},
            {"role": "user", "content": str(prompt) }
        ],
        max_tokens=150,
        temperature=0.7,
    )

print(response.choices[0].message.content.strip())


2. Transfer Learning

In [ ]:
from transformers import BertForSequenceClassification, BertTokenizer
from datasets import load_dataset

model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)

# Load and preprocess dataset
dataset = load_dataset('glue', 'sst2', split='train')
dataset_validation = load_dataset('glue', 'sst2', split='validation')

def preprocess(data):
    return tokenizer(data['sentence'], truncation=True, padding='max_length', max_length=61, return_tensors='pt')

dataset = dataset.map(preprocess, batched=True)
dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
dataset_validation = dataset_validation.map(preprocess, batched=True)
dataset_validation.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# Fine-tune on the new task
from transformers import Trainer, TrainingArguments
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    eval_dataset=dataset_validation,
)

trainer.train()

eval_results = trainer.evaluate()
print(f"Evaluation results for task {i+1}: {eval_results}")

3. Continual Learning

In [ ]:
from transformers import BertForSequenceClassification, BertTokenizer, Trainer, TrainingArguments
from datasets import load_dataset

model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)

# Simulate continual learning with multiple tasks
datasets = [load_dataset('glue', 'sst2', split='train'), load_dataset('glue', 'mrpc', split='train')]
datasets_validation = [load_dataset('glue', 'sst2', split='validation'), load_dataset('glue', 'mrpc', split='validation')]

def preprocess(data):
    if 'sentence' in data:
        return tokenizer(data['sentence'], truncation=True, return_overflowing_tokens=False, padding='max_length', max_length=61, return_tensors='pt')
    else:
        return tokenizer(data['sentence1'], data['sentence2'], truncation=True, return_overflowing_tokens=False, padding='max_length', max_length=61, return_tensors='pt')

for i in range(0,len(datasets)):
    dataset = datasets[i].map(preprocess, batched=True)
    dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
    dataset_validation = datasets_validation[i].map(preprocess, batched=True)
    dataset_validation.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        eval_dataset=dataset_validation
    )

    trainer.train()

    eval_results = trainer.evaluate()
    print(f"Evaluation results for task {i+1}: {eval_results}")

4. Meta-learning

In [ ]:
# Meta-learning frameworks like MAML (Model-Agnostic Meta-Learning) require specialized libraries and more complex setups.

Future Trends and Research Directions

2. Federated Learning:

In [ ]:
      # Federated learning involves complex setups and frameworks like PySyft for implementation.

3. Energy-Efficient Models

In [ ]:
# Techniques involve model pruning, quantization, and efficient data loading, as discussed in previous sections.

Practical Examples

1. Transfer Learning with BERT:

In [ ]:
from transformers import BertForSequenceClassification, BertTokenizer, Trainer, TrainingArguments
from datasets import load_dataset

# Load pre-trained model and tokenizer
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)

# Freeze all layers except for the classification layer
for param in model.bert.parameters():
    param.requires_grad = False

# Load datasets
dataset = load_dataset('glue', 'sst2', split='train')
dataset_validation = load_dataset('glue', 'sst2', split='validation')

# Preprocess the data
def preprocess(data):
    return tokenizer(data['sentence'], truncation=True, padding='max_length', max_length=128)

dataset = dataset.map(preprocess, batched=True)
dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
dataset_validation = dataset_validation.map(preprocess, batched=True)
dataset_validation.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
)

optimizer = AdamW([
    {'params': model.bert.parameters(), 'lr': 1e-5},  # Lower learning rate for frozen layers
    {'params': model.classifier.parameters(), 'lr': 2e-5}  # Higher learning rate for the classifier
])


# Define the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    eval_dataset=dataset_validation,
    optimizers=(optimizer, None)  # Pass optimizer (scheduler can be None or a custom scheduler)
)

# Train the model
trainer.train()

# Evaluate the model
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")
